<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>GraphRAG Index Generation</h3>
    The following is an overview of the workflow:
    <ul>
    <li>Uses Microsoft GraphRAG library to construct a GraphRAG index for the synthetically generated dataset:</li>
        <ul>
            <li>Uses reformatted code-to-summary pairs as input</li>
            <li>Uses openai/gpt-oss-20b at the chat model</li>
            <li>Uses intfloat/e5-mistral-7b-instruct as the embedding model</li>
        </ul>
    </li>
    <li>Stores index in LanceDB database backed by Minio bucket</li>
    </ul>
</div>

In [13]:
##############################################
# Imports
##############################################
from minio import Minio
import os
import lancedb
from datasets import load_dataset
import traceback
import subprocess
import tracemalloc
tracemalloc.start()
import nest_asyncio
nest_asyncio.apply()
import utils

In [2]:
##############################################
# Generate GraphRAG index and store in LanceDB
##############################################

def generate_graphrag_index(graphrag_source_path: str,
                            jsonl_source_path: str,
                            baseline_dataset_name: str, 
                            baseline_dataset_source_path: str, 
                            prompt_dataset_name: str = None, 
                            prompt_dataset_source_path: str = None) :

    """
    Splits the provided jsonl file into seprate json files, then
    generates a GraphRAG index from the json files.
    Args:
        graphrag_source_path: The source path used by the GraphRAG index configuration
        jsonl_source_path: The jsonl source file
        baseline_dataset_name: The source dataset name
        baseline_dataset_source_path: The local file path to the source dataset
        prompt_dataset_name: The dataset name associated with the prompt (optional)
        prompt_dataset_source_path: The local file path to the dataset associated with the prompt (optional)
    Returns:
        None
    """

    ##############################################
    # Imports
    ##############################################
    from minio import Minio
    import os
    import lancedb
    from datasets import load_dataset
    import traceback
    import subprocess
    import tracemalloc
    tracemalloc.start()
    import nest_asyncio
    nest_asyncio.apply()
    import utils

    try:

        graph_rag_config_path = f"{graphrag_source_path}/settings.yaml"

        
        os.makedirs(f"{graphrag_source_path}/input", exist_ok=True)

        os.makedirs(f"{graphrag_source_path}/output", exist_ok=True)
        
        os.makedirs(os.path.dirname(jsonl_source_path), exist_ok=True)
        
        os.makedirs(os.path.dirname(baseline_dataset_source_path), exist_ok=True)

        if prompt_dataset_source_path:
        
            os.makedirs(os.path.dirname(prompt_dataset_source_path), exist_ok=True)
        
    
        updated_dataset = utils.postprocess_dataset(baseline_dataset_name, 
                                                    baseline_dataset_source_path, 
                                                    jsonl_source_path,
                                                    prompt_dataset_name,
                                                    prompt_dataset_source_path)
    
        utils.split_jsonl_into_json_files(jsonl_source_path, f"{graphrag_source_path}/input")
        
    
        result = subprocess.run(["bash", "graphrag.sh", graphrag_source_path, graph_rag_config_path], capture_output=True, text=True, check=False)
            
        print(f"\nSubprocess output: {result.stdout}")
        
        if result.stderr:
            
            raise Exception(f"Error processing GraphRAG command: {result.stderr}")
        
    except Exception as e:
        
        print(f"Error processing GraphRAG DB: {e}")
        traceback.print_exc()

In [3]:
##############################################
# Upload data to LanceDB
##############################################
def upload_graphrag_index_to_lancedb(graphrag_source_path: str, lancedb_minio_bucket_name: str, lancedb_db_name: str):
    """
    Uploads the GraphRAG index from the provided source path to the specified minio bucket.
    (Requires a valid Minio configuration which has been preconfigured using environment variables.)
    Args:
        graphrag_source_path: The source path for the GraphRAG index files
        lancedb_minio_bucket_name: The backing Minio bucket for the LanceDB database.
        lancedb_db_name: The LanceDB database name.
    Returns:
        None
    """

    ##############################################
    # Imports
    ##############################################
    from minio import Minio
    import os
    import lancedb
    from datasets import load_dataset
    import nest_asyncio
    import pandas as pd
    nest_asyncio.apply()
    
    
    try:
        
        graphrag_index_source_path = f"{graphrag_source_path}/output"
        
        db = lancedb.connect(f"s3://{lancedb_minio_bucket_name}/{lancedb_db_name}",
                             
            storage_options={
                "endpoint_url": os.getenv("AWS_S3_ENDPOINT"),
                
                "aws_access_key_id": os.getenv("AWS_ACCESS_KEY_ID"),
                
                "aws_secret_access_key": os.getenv("AWS_SECRET_ACCESS_KEY"),
                
                "s3_force_path_style": "true",
                
                "allow_http": "true",
            }
        )
    
        local_db = lancedb.connect(f"{graphrag_source_path}/output/lancedb")
    
        all_tables = local_db.table_names()
    
        # Migrate Global Search tables
        print("Migrating global search tables...")
        
        for table_name in all_tables:
    
            try:
    
                local_table = local_db.open_table(table_name)
        
                local_df = local_table.to_pandas()
        
                db.create_table(table_name, data=local_df)
        
                print(f"{local_table} migrated.")
        
            except Exception as e:
                
                print(f"Error processing GraphRAG migration to Minio: {e}") 
    
        # Migrate Local Search tables
        print("Migrating local search tables...")
        
        for file_path in os.listdir(f"{graphrag_index_source_path}"):
            
            if file_path.endswith(".parquet"):
    
                try:
            
                    full_path = os.path.join(graphrag_index_source_path, file_path)
        
                    local_df = pd.read_parquet(full_path)
    
                    table_name = file_path.split(".", 1)[0]
        
                    db.create_table(table_name, data=local_df)
        
                    print(f"{table_name} migrated.")
        
                except Exception as e:
                    
                    print(f"Error processing GraphRAG migration to Minio: {e}")    
            
        print("Migration complete.")
        
    except Exception as e:
        
        print(f"Error processing GraphRAG migration to Minio: {e}")

### Generate GraphDB Index and Store in LanceDB
Start the pipeline!

In [4]:
graphrag_source_path = "graph_rag/source"

jsonl_source_path = "graph_rag/json/graphrag.jsonl"

# baseline_dataset_name = "oaawofolu/emerson"

# baseline_dataset_source_path = "json/data.jsonl"

# prompt_dataset_name = "oaawofolu/cfcode-golfap"

# prompt_dataset_source_path = "cfcode/data.jsonl"

baseline_dataset_name = "oaawofolu/cfcode-golfap"

baseline_dataset_source_path = "cfcode/data.jsonl"

bucket_name = "data"

# lancedb_db_name = "cfcode-golfap"

lancedb_db_name = "cfcode-golfap-idx"

# generate_graphrag_index(graphrag_source_path,
#                         jsonl_source_path,
#                         baseline_dataset_name, 
#                         baseline_dataset_source_path, 
#                         prompt_dataset_name, 
#                         prompt_dataset_source_path)


generate_graphrag_index(graphrag_source_path,
                        jsonl_source_path,
                        baseline_dataset_name, 
                        baseline_dataset_source_path)

upload_graphrag_index_to_lancedb(graphrag_source_path, bucket_name, lancedb_db_name)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


KeyboardInterrupt



In [15]:
os.environ["prompt1"] = """
This code represents an application called Golfap, which is 
a social golf scorekeeping app that makes it easy to track scores, bets, and games with friends.

Your task is to analyze this code and generate a software design document.
This document should include a concise explanation of the purpose of the code.
If the provided snippet does not appear to be valid code, indicate that this is not valid code.
Stop after you have finished writing the 3 sections described below.
Do not stray from the functionality provided in the codebase. Stick strictly to the code provided.


Your analysis must include the following sections:

**1. Summary:** 
* Provide a clear and concise explanation of the purpose of the code.

**2. Components:** 
* Provide a concise list of all the important ColdFusion relevant components you can find. 
* Do not rename any components or change their case; provide them exactly as they are shown in the code.
* Include the following:
    * List of primary modules and their responsibilities: include the file path to the modules if it is available
    * List of primary web pages and their responsibilitites: include the file path to the web pages if it is available
    * List of ColdFusion components and their responsibilities: include the file path to the components if it is available
    * List of reusable templates and their responsibilities: include the file path to the templates if it is available
    * List of the names of custom components and their responsibilities (if any), exactly as shown in the code: cfcomponents, user-defined functions and any other custom components you can find; include the file path to the templates if it is available
    
**3. Domain:** 
* Generate a concise outline of the domain model associated with this code.
* Include the current state of the domain objects based on information extracted from the code.

"""

os.environ["prompt2"] = """
This code represents an application called Golfap, which is 
a social golf scorekeeping app that makes it easy to track scores, bets, and games with friends.
Generate a language-agnostic Software Design Document for this code. 
The document should be structured, professional, and suitable for both technical and non-technical stakeholders.
Also exclude any details that link the requirements to ColdFusion or any other specific programming language or framework.
Instead, it should focus on universal concepts and architecture that can be implemented in any programming language.
Stop after you have finished writing the 3 sections described below.

The SDD must include the following sections:

**1. System Architecture**
*   **Overall Design:** Describe the main architectural patterns used and how the different components interact. 
*   **Key Components:** Detail the primary modules, classes, domain model and services and their responsibilities.

**2. Functional Requirements**
*   **Input Handling:** How does the system accept inputs?
*   **Data Processing:** Detail the main logic, algorithms, and data transformations.
*   **Output Generation:** How are results produced and presented?

**3. Business Requirements**
*   **Business Rules:** Specific rules that govern how the business operates, which the software must enforce.
*   **Success Criteria:** Measurable criteria to determine if the project is successful, also known as acceptance criteria.

Only use the context provided in the summary above. Do not stray from the context provided in the summary. 
Include named components and objects based on the context where it makes sense.
"""

In [20]:
! graphrag query --root graph_rag/source --config graph_rag/source/settings.yaml --method global --query "${prompt1}" --response-type "JSON format"

```json
{
  "Summary": "Golfap is a social golf score‑keeping web application built on ColdFusion. It lets golfers record round scores, place bets, and manage games with friends. The code base renders dynamic pages, serves AJAX endpoints for tournament and player data, serializes data to JSON for client‑side consumption, and aggregates news feeds. Google Analytics and AdSense are integrated for usage tracking and monetization. The application is structured around a set of ColdFusion modules, web pages, components, and reusable templates that together provide a responsive, real‑time user experience for golf enthusiasts.",
  "Components": {
    "PrimaryModules": [
      {
        "Name": "m_qtournaments.cfm",
        "Path": "pgatour/tournaments/m_qtournaments.md",
        "Responsibilities": "Retrieves tournament, event, and golfer history data via nested CFQUERY tags; supplies data to AJAX calls for tournament selection.",
        "References": "[Data: Reports (15, 105, 271)]"
      },

In [1]:
json_answer = """
```json
{
  "Summary": "Golfap is a social golf score‑keeping web application built on ColdFusion. It lets golfers record round scores, place bets, and manage games with friends. The code base renders dynamic pages, serves AJAX endpoints for tournament and player data, serializes data to JSON for client‑side consumption, and aggregates news feeds. Google Analytics and AdSense are integrated for usage tracking and monetization. The application is structured around a set of ColdFusion modules, web pages, components, and reusable templates that together provide a responsive, real‑time user experience for golf enthusiasts.",
  "Components": {
    "PrimaryModules": [
      {
        "Name": "m_qtournaments.cfm",
        "Path": "pgatour/tournaments/m_qtournaments.md",
        "Responsibilities": "Retrieves tournament, event, and golfer history data via nested CFQUERY tags; supplies data to AJAX calls for tournament selection.",
        "References": "[Data: Reports (15, 105, 271)]"
      },
      {
        "Name": "ajax_getplayerdata.cfm",
        "Path": "ajax_getplayerdata.cfm",
        "Responsibilities": "Serves AJAX requests for player statistics and tournament results; returns JSON payloads.",
        "References": "[Data: Reports (101, 202, 275)]"
      },
      {
        "Name": "tourney_results.cfm",
        "Path": "tourney_results.cfm",
        "Responsibilities": "Renders tournament result tables and handles score display logic.",
        "References": "[Data: Reports (16, 20, 101, 202, 275)]"
      }
    ],
    "PrimaryWebPages": [
      {
        "Name": "index.cfm",
        "Path": "index.cfm",
        "Responsibilities": "Main entry point; includes header and navigation templates; populates tournament selector and news widgets.",
        "References": "[Data: Reports (169, 5, 17, 111, 22)]"
      },
      {
        "Name": "tournament.cfm",
        "Path": "tournament.cfm",
        "Responsibilities": "Displays detailed tournament information and score tables; uses CFQUERY to fetch data.",
        "References": "[Data: Reports (334, 147, 220, 46, 35)]"
      },
      {
        "Name": "news.cfm",
        "Path": "news.cfm",
        "Responsibilities": "Aggregates golf news from feeds, categories, and stories; serializes to JSON for client widgets.",
        "References": "[Data: Reports (239, 58, 103, 240, 229)]"
      },
      {
        "Name": "leaderboard.cfm",
        "Path": "leaderboard.cfm",
        "Responsibilities": "Shows leaderboard standings; integrates Google Analytics tracking.",
        "References": "[Data: Reports (340, 338)]"
      }
    ],
    "ColdFusionComponents": [
      {
        "Name": "JSON_COMPONENT",
        "Path": "JSON_COMPONENT.cfc",
        "Responsibilities": "Encodes server‑side structures into JSON for AJAX responses.",
        "References": "[Data: Reports (94, 202, 275)]"
      },
      {
        "Name": "USERSESSION",
        "Path": "USERSESSION.cfc",
        "Responsibilities": "Manages session state and admin control visibility.",
        "References": "[Data: Reports (13, 28, 250, 84, 298)]"
      },
      {
        "Name": "ADMINCONTROL",
        "Path": "ADMINCONTROL.cfc",
        "Responsibilities": "Provides UI elements for editing/deleting content and admin actions.",
        "References": "[Data: Reports (13, 28, 250, 84, 298)]"
      }
    ],
    "ReusableTemplates": [
      {
        "Name": "HEADER.CFM",
        "Path": "HEADER.CFM",
        "Responsibilities": "Shared header markup, navigation links, and analytics script inclusion.",
        "References": "[Data: Reports (123, 122, 308)]"
      },
      {
        "Name": "TABS.CFM",
        "Path": "TABS.CFM",
        "Responsibilities": "Renders navigation tabs for news, leaderboard, and tournament pages.",
        "References": "[Data: Reports (107, 216, 100, 206, 183, 190)]"
      },
      {
        "Name": "WIDGET_COL_0",
        "Path": "WIDGET_COL_0.cfm",
        "Responsibilities": "Defines layout column for news and tournament widgets.",
        "References": "[Data: Reports (123, 122, 308)]"
      }
    ],
    "CustomComponents": [
      {
        "Name": "cfcomponents",
        "Path": "cfcomponents.cfc",
        "Responsibilities": "User‑defined functions such as ENCODE, VALIDATE, and custom JSON handling.",
        "References": "[Data: Reports (183)]"
      },
      {
        "Name": "XILINUS.PORTAL",
        "Path": "XILINUS.PORTAL.cfc",
        "Responsibilities": "Manages dynamic widget placement, drag‑drop, and resizing on the portal page.",
        "References": "[Data: Reports (15, 105, 271)]"
      }
    ]
  },
  "Domain": {
    "Entities": [
      {
        "Name": "Golfer",
        "Attributes": ["id", "first_name", "last_name", "handicap"],
        "Relationships": "Linked to Score, Bet, and GolferHistory via golfer_id."
      },
      {
        "Name": "Tournament",
        "Attributes": ["id", "name", "start_date", "course"],
        "Relationships": "Contains multiple Events; associated with Scores and Bets."
      },
      {
        "Name": "Event",
        "Attributes": ["id", "year", "dates", "course"],
        "Relationships": "Joined to Tournament via event_id; linked to GolferHistory."
      },
      {
        "Name": "Score",
        "Attributes": ["player_id", "tournament_id", "round", "strokes", "total"],
        "Relationships": "Belongs to Golfer and Tournament."
      },
      {
        "Name": "Bet",
        "Attributes": ["id", "user_id", "opponent_id", "amount", "outcome"],
        "Relationships": "Associates two Golfer users on a Tournament outcome."
      },
      {
        "Name": "Game",
        "Attributes": ["id", "tournament_id", "status"],
        "Relationships": "Aggregates multiple Scores and Bets."
      },
      {
        "Name": "NewsArticle",
        "Attributes": ["id", "feed_id", "title", "content"],
        "Relationships": "Belongs to FeedSource."
      },
      {
        "Name": "FeedSource",
        "Attributes": ["id", "url", "type"],
        "Relationships": "Provides multiple NewsArticles."
      },
      {
        "Name": "Category",
        "Attributes": ["id", "name"],
        "Relationships": "Classifies FeedSources."
      },
      {
        "Name": "UserSession",
        "Attributes": ["session_id", "user_id", "is_admin"],
        "Relationships": "Controls visibility of ADMINCONTROL elements."
      }
    ],
    "Relationships": "Golfer ↔ Score ↔ Tournament; Tournament ↔ Event ↔ GolferHistory; Bet ↔ Golfer; Game aggregates Bet and Score; NewsArticle ↔ FeedSource ↔ Category.",
    "CurrentState": "All entities are persisted in a relational database accessed via CFQUERY tags. JSON_COMPONENT serializes data for client consumption. The application exposes AJAX endpoints for tournament and player data, and renders pages using reusable templates. Custom components provide additional business logic and UI behavior."
  }
}
```
"""

user_prompt = f"""
Here is a payload containing json content:

{json_answer}

Extract the json content from this payload. For each element in the extracted JSON, output the following:

***Path***: The path to the described item.
***Responsibilities:** The field that describes the responsibilities of the item.

Return the results as a list of JSON objects.
"""

import utils
from dotenv import load_dotenv
import os
import re
import pprint
import json
load_dotenv()
output = utils.simple_llm_tool(user_prompt)
pprint.pprint(output)
# json.loads(re.search(r"```json(.*?)```", output, re.DOTALL).group(1))


('```json\n'
 '[\n'
 '  {\n'
 '    "Path": "pgatour/tournaments/m_qtournaments.md",\n'
 '    "Responsibilities": "Retrieves tournament, event, and golfer history '
 'data via nested CFQUERY tags; supplies data to AJAX calls for tournament '
 'selection."\n'
 '  },\n'
 '  {\n'
 '    "Path": "ajax_getplayerdata.cfm",\n'
 '    "Responsibilities": "Serves AJAX requests for player statistics and '
 'tournament results; returns JSON payloads."\n'
 '  },\n'
 '  {\n'
 '    "Path": "tourney_results.cfm",\n'
 '    "Responsibilities": "Renders tournament result tables and handles score '
 'display logic."\n'
 '  },\n'
 '  {\n'
 '    "Path": "index.cfm",\n'
 '    "Responsibilities": "Main entry point; includes header and navigation '
 'templates; populates tournament selector and news widgets."\n'
 '  },\n'
 '  {\n'
 '    "Path": "tournament.cfm",\n'
 '    "Responsibilities": "Displays detailed tournament information and score '
 'tables; uses CFQUERY to fetch data."\n'
 '  },\n'
 '  {\n'
 '    "P

In [ ]:
# !python3 -m graphrag query --root graph_rag/source/output --method global prompt
! graphrag query --root graph_rag/source --config graph_rag/source/settings.yaml --method global --query "${prompt1}"
# ! graphrag query --help